# Supplementary Figure S10 — does F412L alter access to the L403A-like E423–N179 shift?

This specificity control asks whether the elongated E423–N179 pore–VSD/S6 coordinate is confined to L403A or is also sampled by the other Kv2.1 mutant, F412L. The primary comparison is **WT masked versus F412L masked**, with WT and F412L vanilla ensembles retained as the baseline context.

The threshold (12.841 Å) remains the experimentally derived L403A-like definition: the midpoint between the longest 8SD3 WT subunit and the shortest elongated 8SDA L403A subunit. It is used here as a common structural coordinate, not as an F412L-specific experimental target. Because chain labels in a homotetramer can rotate, subunit distances are ranked within each structure before the asymmetry pattern is compared.

Inference uses complete input seeds as the independent units. Recycles are reduced within each seed–AF2-model trajectory, available AF2 models receive equal weight within a seed, and seeds receive equal weight between groups.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

repo_root = Path.cwd()
if not (repo_root / 'shared').is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from shared.distribution_statistics import parse_trajectory_metadata
from shared.data_access import resolve
from shared.dataset_selection import (
    distance_csv_options, load_selected_distance_csv,
    kv21_active_masked_distance_path,
)
from shared.plotting import (
    KV21_PALETTE, apply_kv21_style, experimental_reference_style,
)

apply_kv21_style()

FIG_DIR = repo_root / 'docs/figures/supplementary_figure_s10'
TAB_DIR = repo_root / 'docs/tables/supplementary_figure_s10'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_PATHS = {
    ('WT', 'vanilla'): Path('kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF2_distances_all.csv'),
    ('WT', 'masked'): kv21_active_masked_distance_path(repo_root, 'WT'),
    ('F412L', 'vanilla'): Path('kv21/dataDistances/26-02-11_Kv2.1_f412l_vanillaAF2_distances_all.csv'),
    ('F412L', 'masked'): kv21_active_masked_distance_path(repo_root, 'F412L'),
}
DATASETS = {
    key: load_selected_distance_csv(
        f'{key[0]} {key[1]}',
        distance_csv_options(repo_root, path, 'kv21', key[0], key[1]),
        'all_ok_3_structural_interface_qc',
        fallback_to_all=False,
    )
    for key, path in ORIGINAL_PATHS.items()
}
DISTANCE_COLUMNS = [f'CA_CA_{chain}_GLU425_CA-{chain}_ASN181_CA' for chain in 'ABCD']
GROUP_ORDER = [('WT', 'vanilla'), ('WT', 'masked'), ('F412L', 'vanilla'), ('F412L', 'masked')]
GROUP_LABELS = {
    ('WT', 'vanilla'): 'WT | vanilla', ('WT', 'masked'): 'WT | masked',
    ('F412L', 'vanilla'): 'F412L | vanilla', ('F412L', 'masked'): 'F412L | masked',
}
GROUP_COLORS = {
    ('WT', 'vanilla'): KV21_PALETTE['WT_VAN'],
    ('WT', 'masked'): KV21_PALETTE['WT_HM'],
    ('F412L', 'vanilla'): KV21_PALETTE['F412L_VAN'],
    ('F412L', 'masked'): KV21_PALETTE['F412L_HM'],
}
BASE_SEED = 20260824
BOOTSTRAP_REPLICATES = 10_000

Merged companion distance table: kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF2_distances_extra_all_ok_rmsd_3A_structural_interface_qc.csv
WT vanilla: requested=all_ok_3_structural_interface_qc; actual=all_ok_3_structural_interface_qc; path=kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF2_distances_all_ok_rmsd_3A_structural_interface_qc.csv
Merged companion distance table: kv21/dataDistances/26-11-20_Kv2.1_wt_masked_common99_50pct_AF2_distances_extra_all_ok_rmsd_3A_structural_interface_qc.csv
WT masked: requested=all_ok_3_structural_interface_qc; actual=all_ok_3_structural_interface_qc; path=kv21/dataDistances/26-11-20_Kv2.1_wt_masked_common99_50pct_AF2_distances_all_ok_rmsd_3A_structural_interface_qc.csv


Merged companion distance table: kv21/dataDistances/26-02-11_Kv2.1_f412l_vanillaAF2_distances_extra_all_ok_rmsd_3A_structural_interface_qc.csv
F412L vanilla: requested=all_ok_3_structural_interface_qc; actual=all_ok_3_structural_interface_qc; path=kv21/dataDistances/26-02-11_Kv2.1_f412l_vanillaAF2_distances_all_ok_rmsd_3A_structural_interface_qc.csv
Merged companion distance table: kv21/dataDistances/26-11-20_Kv2.1_f412l_masked_common99_50pct_AF2_distances_extra_all_ok_rmsd_3A_structural_interface_qc.csv
F412L masked: requested=all_ok_3_structural_interface_qc; actual=all_ok_3_structural_interface_qc; path=kv21/dataDistances/26-11-20_Kv2.1_f412l_masked_common99_50pct_AF2_distances_all_ok_rmsd_3A_structural_interface_qc.csv


## Convergence-qualified cohorts and experimental definition

All four ensembles use the explicit convergence-qualified all_ok_rmsd_3A tables. The CSV residue labels are offset by +2 relative to experimental numbering; GLU425–ASN181 therefore reports experimental E423–N179. The structural/interface-QC reconciliation is uploaded separately and is not mixed into this figure.

In [2]:
def experimental_ca_distances(path):
    coordinates = {}
    for line in path.read_text().splitlines():
        if not line.startswith('ATOM') or line[12:16].strip() != 'CA':
            continue
        residue, chain = line[17:20].strip(), line[21].strip()
        try:
            number = int(line[22:26])
        except ValueError:
            continue
        if (residue, number) not in {('GLU', 423), ('ASN', 179)}:
            continue
        coordinates[(chain, residue, number)] = np.array([
            float(line[30:38]), float(line[38:46]), float(line[46:54])
        ])
    return {
        chain: float(np.linalg.norm(
            coordinates[(chain, 'GLU', 423)] - coordinates[(chain, 'ASN', 179)]
        ))
        for chain in 'ABCD'
    }

experimental_values = {
    '8SD3': experimental_ca_distances(repo_root / 'kv21/experimental/8SD3.pdb'),
    '8SDA': experimental_ca_distances(repo_root / 'kv21/experimental/8SDA.pdb'),
}
wt_max = max(experimental_values['8SD3'].values())
shifted_values = sorted(value for value in experimental_values['8SDA'].values() if value > wt_max)
if len(shifted_values) != 2:
    raise ValueError(f'Expected two 8SDA distances above the WT maximum; got {shifted_values}')
SHIFT_THRESHOLD = (wt_max + shifted_values[0]) / 2
experimental_ranked = {
    structure: np.sort(list(values.values()))
    for structure, values in experimental_values.items()
}
pd.DataFrame([
    {'structure': structure, 'chain': chain, 'E423_N179_CA_distance_A': value,
     'derived_threshold_A': SHIFT_THRESHOLD}
    for structure, values in experimental_values.items()
    for chain, value in values.items()
]).to_csv(TAB_DIR / 'l403a_experimental_threshold_derivation.csv', index=False)
print(f'Experiment-anchored threshold: {SHIFT_THRESHOLD:.6f} Å')

frames = {}
audit_rows = []
for key, source in DATASETS.items():
    frame = source[['pdb_file', *DISTANCE_COLUMNS]].copy()
    frame = parse_trajectory_metadata(frame)
    frame[DISTANCE_COLUMNS] = frame[DISTANCE_COLUMNS].apply(pd.to_numeric, errors='coerce')
    if frame[DISTANCE_COLUMNS].isna().any().any():
        raise ValueError(f'Missing E423–N179 values in {key}')
    ranked = np.sort(frame[DISTANCE_COLUMNS].to_numpy(float), axis=1)
    for index in range(4):
        frame[f'rank_{index + 1}_A'] = ranked[:, index]
    frame['maximum_distance_A'] = ranked[:, -1]
    frame['shifted_subunits'] = (ranked >= SHIFT_THRESHOLD).sum(axis=1)
    frame['any_shifted'] = frame.shifted_subunits.ge(1).astype(float)
    for number in range(1, 5):
        frame[f'exactly_{number}_shifted'] = frame.shifted_subunits.eq(number).astype(float)
    frames[key] = frame
    audit_rows.append({
        'sequence': key[0], 'protocol': key[1],
        'source_path': str(source.attrs.get('source_path', 'in-memory selected table')),
        'snapshot_rows': len(frame), 'retained_seeds': frame.seed.nunique(),
        'retained_seed_model_trajectories': frame[['seed', 'model_number']].drop_duplicates().shape[0],
        'threshold_A': SHIFT_THRESHOLD,
    })

cohort_audit = pd.DataFrame(audit_rows)
cohort_audit.to_csv(TAB_DIR / 'e423_n179_cohort_audit.csv', index=False)
display(cohort_audit)

Experiment-anchored threshold: 12.841337 Å


,sequence,protocol,source_path,snapshot_rows,retained_seeds,retained_seed_model_trajectories,threshold_A
0,WT,vanilla,kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF...,4591,100,490,12.841337
1,WT,masked,kv21/dataDistances/26-11-20_Kv2.1_wt_masked_co...,1529,100,219,12.841337
2,F412L,vanilla,kv21/dataDistances/26-02-11_Kv2.1_f412l_vanill...,4521,100,487,12.841337
3,F412L,masked,kv21/dataDistances/26-11-20_Kv2.1_f412l_masked...,2543,100,320,12.841337


## Seed-balanced summaries and contrasts

The continuous endpoint is the maximum E423–N179 distance across the tetramer: each trajectory contributes its median retained recycle, AF2 models are averaged within seed, and seeds are averaged between groups. Shifted-state endpoints first calculate the fraction of retained snapshots within each trajectory and then apply the same model-within-seed weighting.

Confidence intervals are percentile intervals from 10,000 independent whole-seed bootstrap draws. Identical numerical seed labels are not treated as paired observations across separate prediction runs.

In [3]:
def seed_summary(frame, column, reduction):
    grouped = frame.groupby(['seed', 'model_number'])[column]
    trajectories = grouped.median() if reduction == 'median' else grouped.mean()
    return trajectories.groupby('seed').mean().sort_index()


metric_definitions = {
    'maximum_distance_A': ('maximum_distance_A', 'median', 'Å'),
    'any_shifted_fraction': ('any_shifted', 'mean', 'fraction'),
    'exactly_two_shifted_fraction': ('exactly_2_shifted', 'mean', 'fraction'),
    'four_shifted_fraction': ('exactly_4_shifted', 'mean', 'fraction'),
}
seed_values = {
    key: {
        metric: seed_summary(frame, column, reduction)
        for metric, (column, reduction, _) in metric_definitions.items()
    }
    for key, frame in frames.items()
}

# Ranked subunit values for the masked-only distribution panel.
rank_seed_values = {}
for key in [('WT', 'masked'), ('F412L', 'masked')]:
    rank_seed_values[key] = {
        rank_number: seed_summary(frames[key], f'rank_{rank_number}_A', 'median')
        for rank_number in range(1, 5)
    }

bootstrap_draws = {}
summary_rows = []
for group_index, key in enumerate(GROUP_ORDER):
    bootstrap_draws[key] = {}
    for metric_index, (metric, (_, reduction, unit)) in enumerate(metric_definitions.items()):
        values = seed_values[key][metric].to_numpy(float)
        rng = np.random.default_rng(BASE_SEED + 100 * group_index + metric_index)
        draws = values[rng.integers(0, len(values), size=(BOOTSTRAP_REPLICATES, len(values)))].mean(axis=1)
        bootstrap_draws[key][metric] = draws
        summary_rows.append({
            'sequence': key[0], 'protocol': key[1], 'metric': metric,
            'seed_balanced_estimate': values.mean(),
            'ci_low': np.quantile(draws, .025), 'ci_high': np.quantile(draws, .975),
            'unit': unit, 'within_trajectory_reduction': reduction,
            'retained_seeds': len(values), 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        })
group_summary = pd.DataFrame(summary_rows)

contrast_definitions = {
    'F412L_minus_WT__masked': (('WT', 'masked'), ('F412L', 'masked')),
    'F412L_minus_WT__vanilla': (('WT', 'vanilla'), ('F412L', 'vanilla')),
    'masked_minus_vanilla__WT': (('WT', 'vanilla'), ('WT', 'masked')),
    'masked_minus_vanilla__F412L': (('F412L', 'vanilla'), ('F412L', 'masked')),
}
contrast_rows = []
for metric, (_, _, unit) in metric_definitions.items():
    for label, (group_a, group_b) in contrast_definitions.items():
        samples = bootstrap_draws[group_b][metric] - bootstrap_draws[group_a][metric]
        contrast_rows.append({
            'metric': metric, 'contrast': label,
            'estimate': seed_values[group_b][metric].mean() - seed_values[group_a][metric].mean(),
            'ci_low': np.quantile(samples, .025), 'ci_high': np.quantile(samples, .975),
            'unit': unit, 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        })
    interaction_samples = (
        bootstrap_draws[('F412L', 'masked')][metric]
        - bootstrap_draws[('F412L', 'vanilla')][metric]
        - bootstrap_draws[('WT', 'masked')][metric]
        + bootstrap_draws[('WT', 'vanilla')][metric]
    )
    interaction_estimate = (
        seed_values[('F412L', 'masked')][metric].mean()
        - seed_values[('F412L', 'vanilla')][metric].mean()
        - seed_values[('WT', 'masked')][metric].mean()
        + seed_values[('WT', 'vanilla')][metric].mean()
    )
    contrast_rows.append({
        'metric': metric, 'contrast': 'masking_by_F412L_interaction',
        'estimate': interaction_estimate,
        'ci_low': np.quantile(interaction_samples, .025),
        'ci_high': np.quantile(interaction_samples, .975),
        'unit': unit, 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    })
contrasts = pd.DataFrame(contrast_rows)

# Full shifted-subunit occupancy for the two masked ensembles.
occupancy_rows = []
occupancy_draws = {}
for group_index, key in enumerate([('WT', 'masked'), ('F412L', 'masked')]):
    occupancy_draws[key] = {}
    for number in range(1, 5):
        values = seed_summary(frames[key], f'exactly_{number}_shifted', 'mean').to_numpy(float)
        rng = np.random.default_rng(BASE_SEED + 1000 + 10 * group_index + number)
        draws = values[rng.integers(0, len(values), size=(BOOTSTRAP_REPLICATES, len(values)))].mean(axis=1)
        occupancy_draws[key][number] = draws
        occupancy_rows.append({
            'sequence': key[0], 'protocol': key[1], 'shifted_subunits': number,
            'seed_balanced_fraction': values.mean(),
            'ci_low': np.quantile(draws, .025), 'ci_high': np.quantile(draws, .975),
        })
occupancy = pd.DataFrame(occupancy_rows)

seed_level_long = pd.concat([
    pd.DataFrame({
        'sequence': key[0], 'protocol': key[1], 'seed': values.index,
        'metric': metric, 'value': values.to_numpy(float),
    })
    for key in GROUP_ORDER for metric, values in seed_values[key].items()
], ignore_index=True)

group_summary.to_csv(TAB_DIR / 'e423_n179_group_summary.csv', index=False)
contrasts.to_csv(TAB_DIR / 'e423_n179_seed_bootstrap_contrasts.csv', index=False)
occupancy.to_csv(TAB_DIR / 'e423_n179_masked_shifted_subunit_occupancy.csv', index=False)
seed_level_long.to_csv(TAB_DIR / 'e423_n179_seed_level_values.csv', index=False)

display(group_summary.pivot(index=['sequence', 'protocol'], columns='metric', values='seed_balanced_estimate').round(4))
display(contrasts[contrasts.contrast.isin(['F412L_minus_WT__masked', 'masking_by_F412L_interaction'])].round(4))

metric             any_shifted_fraction  exactly_two_shifted_fraction  \
sequence protocol                                                       
F412L    masked                  0.5372                        0.0043   
         vanilla                 0.2446                        0.0095   
WT       masked                  0.2296                        0.0031   
         vanilla                 0.0075                        0.0000   

metric             four_shifted_fraction  maximum_distance_A  
sequence protocol                                             
F412L    masked                   0.5237             13.1684  
         vanilla                  0.2176             12.1179  
WT       masked                   0.2253             11.4779  
         vanilla                  0.0058             10.5576

,metric,contrast,estimate,ci_low,ci_high,unit,bootstrap_replicates
0,maximum_distance_A,F412L_minus_WT__masked,1.6905,1.4217,1.9447,Å,10000
4,maximum_distance_A,masking_by_F412L_interaction,0.1302,-0.1543,0.4009,Å,10000
5,any_shifted_fraction,F412L_minus_WT__masked,0.3076,0.2328,0.3798,fraction,10000
9,any_shifted_fraction,masking_by_F412L_interaction,0.0705,-0.0061,0.1467,fraction,10000
10,exactly_two_shifted_fraction,F412L_minus_WT__masked,0.0013,-0.0036,0.0052,fraction,10000
14,exactly_two_shifted_fraction,masking_by_F412L_interaction,-0.0082,-0.0140,-0.0034,fraction,10000
15,four_shifted_fraction,F412L_minus_WT__masked,0.2985,0.2243,0.3716,fraction,10000
19,four_shifted_fraction,masking_by_F412L_interaction,0.0867,0.0101,0.1627,fraction,10000


## Supplementary figure

Panel A compares ranked subunit distances within the two masked ensembles and overlays the ranked experimental structures. Panel B places the maximum tetramer distance in the four-group factorial context. Panel C reports the absolute prevalence of any threshold-crossing subunit. Panel D resolves the masked shifted tail by the number of shifted subunits.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14.2, 10.8), facecolor='white')
fig.subplots_adjust(left=.085, right=.985, top=.935, bottom=.15, hspace=.36, wspace=.26)
ax_a, ax_b, ax_c, ax_d = axes.ravel()

# A: ranked subunit distances in masked WT versus masked F412L.
rank_records = []
for key in [('WT', 'masked'), ('F412L', 'masked')]:
    for rank_number, values in rank_seed_values[key].items():
        rank_records.extend({
            'Sequence': key[0], 'Rank': f'{rank_number}', 'Distance': value,
        } for value in values)
rank_plot = pd.DataFrame(rank_records)
sns.violinplot(
    data=rank_plot, x='Rank', y='Distance', hue='Sequence',
    order=['1', '2', '3', '4'], hue_order=['WT', 'F412L'], split=True,
    inner='quartile', cut=0, linewidth=.8,
    palette={'WT': GROUP_COLORS[('WT', 'masked')], 'F412L': GROUP_COLORS[('F412L', 'masked')]},
    ax=ax_a,
)
for rank_index in range(4):
    for structure, offset in [('8SD3', -.055), ('8SDA', .055)]:
        style = experimental_reference_style(structure)
        ax_a.scatter(
            rank_index + offset, experimental_ranked[structure][rank_index],
            s=42, marker=style['marker'], facecolor='white', edgecolor=style['color'],
            linewidth=1.25, zorder=8,
        )
ax_a.axhline(SHIFT_THRESHOLD, color='#666666', ls='--', lw=1.0)
ax_a.set(
    title='Masked ensembles: ranked subunit distances',
    xlabel='Within-structure E423–N179 rank', ylabel='Seed-balanced E423–N179 Cα distance (Å)',
)
ax_a.get_legend().remove()

# B: maximum distance in all four groups.
max_records = []
for key in GROUP_ORDER:
    max_records.extend({
        'Group': GROUP_LABELS[key], 'Maximum': value,
    } for value in seed_values[key]['maximum_distance_A'])
max_plot = pd.DataFrame(max_records)
label_order = [GROUP_LABELS[key] for key in GROUP_ORDER]
palette = {GROUP_LABELS[key]: GROUP_COLORS[key] for key in GROUP_ORDER}
sns.violinplot(
    data=max_plot, x='Group', y='Maximum', hue='Group', order=label_order,
    palette=palette, inner='quartile', cut=0, linewidth=.8, legend=False, ax=ax_b,
)
ax_b.axhline(SHIFT_THRESHOLD, color='#555555', ls='--', lw=1.1)
ax_b.text(2.98, SHIFT_THRESHOLD + .08, f'shifted threshold = {SHIFT_THRESHOLD:.2f} Å',
          ha='right', va='bottom', fontsize=9.5, color='#555555')
ax_b.set(
    title='Maximum tetramer distance', xlabel='',
    ylabel='Seed-balanced maximum E423–N179 (Å)',
)
ax_b.tick_params(axis='x', rotation=17)

# C: absolute prevalence of any shifted subunit.
any_summary = group_summary[group_summary.metric.eq('any_shifted_fraction')].copy()
any_summary['key'] = list(zip(any_summary.sequence, any_summary.protocol))
any_summary = any_summary.set_index('key').loc[GROUP_ORDER].reset_index()
x = np.arange(4)
y = any_summary.seed_balanced_estimate.to_numpy(float) * 100
lo = any_summary.ci_low.to_numpy(float) * 100
hi = any_summary.ci_high.to_numpy(float) * 100
bars = ax_c.bar(
    x, y, width=.68, color=[GROUP_COLORS[key] for key in GROUP_ORDER],
    edgecolor='#4A4A4A', linewidth=.75,
)
ax_c.errorbar(x, y, yerr=[y - lo, hi - y], fmt='none', ecolor='#333333',
              elinewidth=1.25, capsize=3, zorder=4)
for bar, value in zip(bars, y):
    label = f'{value:.2f}%' if value < 1 else f'{value:.1f}%'
    ax_c.text(bar.get_x() + bar.get_width()/2, value + .45, label,
              ha='center', va='bottom', fontsize=10, fontweight='semibold')
ax_c.set_xticks(x, label_order, rotation=17)
ax_c.set(
    title='Prevalence of the elongated state', xlabel='',
    ylabel='Seed-balanced snapshots with ≥1 shifted subunit (%)',
)
ax_c.set_ylim(0, max(18, hi.max() * 1.22))

# D: number of shifted subunits in the masked tail (unconditional frequencies).
offsets = {'WT': -.17, 'F412L': .17}
for sequence, color in [('WT', GROUP_COLORS[('WT', 'masked')]),
                        ('F412L', GROUP_COLORS[('F412L', 'masked')])]:
    part = occupancy[occupancy.sequence.eq(sequence)].sort_values('shifted_subunits')
    xx = np.arange(1, 5) + offsets[sequence]
    yy = part.seed_balanced_fraction.to_numpy(float) * 100
    lower = part.ci_low.to_numpy(float) * 100
    upper = part.ci_high.to_numpy(float) * 100
    ax_d.bar(xx, yy, width=.32, color=color, edgecolor='#4A4A4A', linewidth=.7)
    ax_d.errorbar(xx, yy, yerr=[yy-lower, upper-yy], fmt='none', ecolor='#333333',
                  elinewidth=1.15, capsize=2.5, zorder=4)
ax_d.axvline(2, color=experimental_reference_style('8SDA')['color'], lw=1.1, ls=':')
ax_d.text(2.04, 41.0, '8SDA: 2 shifted subunits', rotation=90, ha='left', va='top',
          fontsize=9.2, color=experimental_reference_style('8SDA')['color'])
ax_d.set_xticks(range(1, 5))
ax_d.set(
    title='Composition of the masked shifted tail', xlabel='Shifted subunits per structure',
    ylabel='Seed-balanced snapshot fraction (%)',
)
ax_d.set_ylim(0, max(44, float(occupancy.ci_high.max()) * 100 * 1.15))

for label, axis in zip('ABCD', axes.ravel()):
    axis.text(-.015, 1.055, label, transform=axis.transAxes, fontsize=16,
              fontweight='bold', ha='left', va='top')
    axis.grid(axis='x', visible=False)
    sns.despine(ax=axis)

wt_exp_style = experimental_reference_style('8SD3')
mut_exp_style = experimental_reference_style('8SDA')
legend_handles = [
    Patch(facecolor=GROUP_COLORS[key], edgecolor='#4A4A4A', label=GROUP_LABELS[key])
    for key in GROUP_ORDER
] + [
    Line2D([0], [0], marker=wt_exp_style['marker'], color='none', markerfacecolor='white',
           markeredgecolor=wt_exp_style['color'], markeredgewidth=1.2, label='8SD3 | WT'),
    Line2D([0], [0], marker=mut_exp_style['marker'], color='none', markerfacecolor='white',
           markeredgecolor=mut_exp_style['color'], markeredgewidth=1.2, label='8SDA | L403A'),
]
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(.535, .025),
           ncol=3, frameon=False, fontsize=10.5)
fig.suptitle(
    r'$\mathrm{K}_{\mathrm{V}}2.1$ | masking exposes the E423–N179 elongated state in WT and F412L',
    fontsize=16.5, fontweight='semibold', y=.982,
)

png = FIG_DIR / 'Figure_S10_Kv21_F412L_masked_WT_vs_mutant.png'
pdf = FIG_DIR / 'Figure_S10_Kv21_F412L_masked_WT_vs_mutant.pdf'
fig.savefig(png, dpi=400, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show(); plt.close(fig)
print('Saved:', png.relative_to(repo_root))
print('Saved:', pdf.relative_to(repo_root))

Saved: docs/figures/supplementary_figure_s10/Figure_S10_Kv21_F412L_masked_WT_vs_mutant.png
Saved: docs/figures/supplementary_figure_s10/Figure_S10_Kv21_F412L_masked_WT_vs_mutant.pdf


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_61236/2457691470.py:132: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


## Take-home message

F412L is evaluated against the same current convergence-qualified E423–N179 coordinate as WT. The exact seed-balanced shifted-state prevalence and uncertainty intervals are written to `e423_n179_group_summary.csv`; no legacy percentages are hard-coded into this notebook.

The masked-versus-WT contrast, the masking response, and the masking-by-F412L interaction are reported in `e423_n179_seed_bootstrap_contrasts.csv`. This keeps the specificity control tied to the current 50% inputs rather than to earlier dataset denominators.

As in the L403A control, the threshold-positive population is dominated by four-subunit elongation rather than the two-subunit asymmetry of experimental 8SDA. E423–N179 elongation is therefore not unique to L403A and should be interpreted as a shared Kv2.1 conformational coordinate whose baseline accessibility is especially high in F412L—not as a mutation-specific reconstruction of 8SDA.